# QR Scan Event Producer

## Purpose of This Notebook

This notebook implements the **event producer** for the anomaly detection pipeline.

Its purpose is to **simulate QR scan events** and inject them into the pipeline **asynchronously**, using Kafka.

This notebook does **not** perform any anomaly detection or analysis.  
It represents the *upstream data source* of the system.

---

## What This Producer Represents

Conceptually, this notebook simulates:

- Mobile device QR code scanning
- URL redirection from QR codes
- Web requests from scanned URLs

In a real system, similar data would come from:

- mobile app logs,
- web server access logs,
- URL scanning services.

Here, the data is **synthetic**, but the **architecture is realistic**.

---

## Key Architectural Ideas Demonstrated

### 1. Event-Driven Design

Events are sent to Kafka **independently of consumers**.

The producer:
- does not know who will process the data,
- does not wait for anomaly detection results,
- does not depend on downstream components.

### 2. Message Queue Instead of Function Calls

Kafka is used instead of direct function calls to ensure:

- buffering under load,
- fault tolerance,
- asynchronous processing,
- scalability.

### 3. Distributed Tracing from the First Stage

Each generated event is assigned:

- a unique `event_id`,
- a corresponding **trace ID**,
- and a producer-side trace span.

This allows the **entire lifecycle of one event** to be observed later in Jaeger,
across multiple pipeline stages.

---

## Generated Event Structure

Each event is a JSON object with fields such as:

- `event_id`
- `timestamp`
- `user`
- `device_id`
- `url`
- `url_length`
- `dot_count`
- `protocol`
- `hour`

This structure represents QR scan telemetry.

---

## Tracing Model Used Here

This producer uses **OpenTelemetry** to emit traces.

Important design choice:

> **The trace ID is derived from the event ID.**

This ensures that:
- producer spans,
- consumer spans,
- and processing spans

all appear as part of **one single trace** in Jaeger.

---

## Anomaly Generation

This producer generates both normal QR scans and anomalies (phishing attempts):

- Normal: Work hours, short URLs, HTTPS
- Anomalous: Late night, very long URLs, many dots, HTTP

The anomaly rate is higher than in training to simulate real-world detection scenarios.

In [ ]:
!pip install -q kafka-python

In [ ]:
!pip install -q opentelemetry-sdk

In [ ]:
!pip install -q opentelemetry-exporter-otlp

In [ ]:
import json
import random
import time
import uuid
import numpy as np
from datetime import datetime, timezone

from kafka import KafkaProducer

# ---------------- OpenTelemetry ----------------
from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import (
    OTLPSpanExporter,
)
from opentelemetry.trace import SpanKind, TraceFlags
from opentelemetry.trace import set_span_in_context
from opentelemetry.trace import SpanContext, NonRecordingSpan

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

KAFKA_BOOTSTRAP_SERVERS = "kafka:9092"
KAFKA_TOPIC = "qr-events"  # Different topic from classification pipeline

# OpenTelemetry Collector endpoint
OTLP_ENDPOINT = "jaeger:4317"  # gRPC
SERVICE_NAME = "qr-scan-producer"  # Different service name

EVENT_INTERVAL_SEC = (0.5, 2)  # Faster events for anomaly detection

# ---------------------------------------------------------------------
# Tracing setup (OTLP)
# ---------------------------------------------------------------------

resource = Resource.create(
    {
        "service.name": SERVICE_NAME,
    }
)

trace.set_tracer_provider(TracerProvider(resource=resource))
tracer = trace.get_tracer(__name__)

otlp_exporter = OTLPSpanExporter(
    endpoint=OTLP_ENDPOINT,
    insecure=True,  # внутри docker-сети
)

span_processor = BatchSpanProcessor(otlp_exporter)
trace.get_tracer_provider().add_span_processor(span_processor)

# ---------------------------------------------------------------------
# Kafka producer
# ---------------------------------------------------------------------

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

# ---------------------------------------------------------------------
# Event generation
# ---------------------------------------------------------------------

USERS = ["alice", "bob", "charlie", "diana", "eve"]
DEVICES = ["phone-001", "phone-002", "tablet-001", "laptop-001"]

def generate_qr_event():
    event_id = str(uuid.uuid4())
    
    # Decide if this is an anomaly (higher rate than training)
    is_anomaly = random.random() < 0.08  # 8% anomalies
    
    if is_anomaly:
        # Anomalous QR scan (phishing attempt)
        hour = random.randint(0, 5)  # Late night
        url_length = random.randint(80, 150)  # Very long URL
        dot_count = random.randint(5, 10)  # Many dots
        protocol = 'http'  # Insecure
        url = f"http://{'x'*url_length}.{'com'*dot_count}/"
    else:
        # Normal QR scan
        hour = random.randint(8, 18)  # Work hours
        url_length = int(np.random.normal(25, 5))  # Normal length
        dot_count = random.randint(1, 3)  # Few dots
        protocol = random.choice(['https', 'http']) if random.random() < 0.1 else 'https'  # Mostly HTTPS
        url = f"{protocol}://example{'x'*url_length}.{'com'*dot_count}/"

    event = {
        "event_id": event_id,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "user": random.choice(USERS),
        "device_id": random.choice(DEVICES),
        "url": url,
        "url_length": url_length,
        "dot_count": dot_count,
        "protocol": protocol,
        "hour": hour,
        "is_anomaly_ground_truth": is_anomaly,  # For evaluation purposes
    }

    return event

# ---------------------------------------------------------------------
# Main loop
# ---------------------------------------------------------------------

print("Starting QR scan event producer (OTLP)...")
print(f"Kafka topic: {KAFKA_TOPIC}")
print(f"OTLP endpoint: {OTLP_ENDPOINT}")
print("Generating QR scan events with 8% anomaly rate...")

while True:
    event = generate_qr_event()
    event_id = event["event_id"]

    # event_id → trace_id (UUID → 128-bit int)
    trace_id = uuid.UUID(event_id).int

    parent_ctx = set_span_in_context(
        NonRecordingSpan(
            SpanContext(
                trace_id=trace_id,
                span_id=random.getrandbits(64),
                is_remote=False,
                trace_flags=TraceFlags(TraceFlags.SAMPLED),
                trace_state={},
            )
        )
    )

    with tracer.start_as_current_span(
        "produce_qr_event",
        context=parent_ctx,
        kind=SpanKind.PRODUCER,
    ) as span:

        span.set_attribute("event.id", event_id)
        span.set_attribute("event.type", "qr_scan")
        span.set_attribute("device.id", event["device_id"])
        span.set_attribute("user.name", event["user"])
        span.set_attribute("url.length", event["url_length"])
        span.set_attribute("is.anomaly", event["is_anomaly_ground_truth"])

        with tracer.start_as_current_span("kafka_produce"):
            producer.send(KAFKA_TOPIC, event)
            producer.flush()

        print(f"Produced QR event {event_id} (anomaly: {event['is_anomaly_ground_truth']})")

    time.sleep(random.uniform(*EVENT_INTERVAL_SEC))

Starting QR scan event producer (OTLP)...
Kafka topic: qr-events
OTLP endpoint: jaeger:4317
Generating QR scan events with 8% anomaly rate...
Produced QR event 3de6069a-e3f7-4d7e-89b3-a10c01251bfc (anomaly: False)
Produced QR event de0dad92-46eb-499f-b455-801d1f3dbd8c (anomaly: False)
Produced QR event 3029a1ea-cfb9-4384-a9af-8a677ed94cd6 (anomaly: False)
Produced QR event 6f316c2b-85e1-4f73-acac-dd27d7528f20 (anomaly: False)
Produced QR event 553fa417-e8f7-4f90-a33c-445489fbee6e (anomaly: True)
Produced QR event 04c4b1b2-2258-46fb-8cc3-6dec99ab7ff7 (anomaly: False)
Produced QR event 52bd57e3-4d35-4652-b6cf-56b286d13acd (anomaly: False)
Produced QR event bbd58309-1035-4a42-8e5d-ef07b8968910 (anomaly: False)
Produced QR event 225a5d59-2af1-4a7f-8a38-8798c4c8afd5 (anomaly: False)
Produced QR event ca01f3fa-aa99-4c68-9700-af4b7ea7277f (anomaly: False)
Produced QR event 70583536-22fb-44eb-a0d7-86d53f4ed404 (anomaly: False)
Produced QR event 3bb86b81-29fa-4f9e-8a04-c1314925f106 (anomaly: Fa